In [ ]:
import os
import asyncio
import winrt.windows.media.ocr as ocr
import winrt.windows.graphics.imaging as imaging
import winrt.windows.storage.streams as streams
import winrt.windows.globalization as globalization

async def run_win_ocr_on_file(ocr_engine, image_path):
    """使用 Windows 原生 OCR 引擎辨識單張圖片（已修正 stream 寫法）"""
    with open(image_path, 'rb') as f:
        img_bytes = f.read()

    # 1. 建立記憶體串流
    stream = streams.InMemoryRandomAccessStream()
    writer = streams.DataWriter(stream)
    writer.write_bytes(img_bytes)
    
    # 2. 將資料寫入並重置流指標到開頭 (Position = 0)
    await writer.store_async()
    await writer.flush_async()
    writer.detach_stream()
    stream.seek(0) if hasattr(stream, 'seek') else setattr(stream, 'position', 0)

    # 3. 建立圖片解碼器與 Bitmap
    decoder = await imaging.BitmapDecoder.create_async(stream)
    software_bitmap = await decoder.get_software_bitmap_async()

    # 4. 進行辨識
    result = await ocr_engine.recognize_async(software_bitmap)
    return result.text


async def main():
    folder_path = r"C:/Users/user/downloads"  # 請替換為你的圖片資料夾路徑
    output_file = "extracted_text.txt"

    # 優先嘗試載入繁體中文，若無則載入系統預設語言
    lang_zh = globalization.Language("zh-Hant-TW")
    if ocr.OcrEngine.is_language_supported(lang_zh):
        ocr_engine = ocr.OcrEngine.try_create_from_language(lang_zh)
        print("已成功載入：繁體中文 OCR 引擎")
    else:
        ocr_engine = ocr.OcrEngine.try_create_from_user_profile_languages()
        print(f"載入系統預設 OCR 引擎：{ocr_engine.recognizer_language.language_tag}")

    valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp')
    
    if not os.path.exists(folder_path):
        print(f"找不到資料夾：{folder_path}，請確認路徑是否正確。")
        return

    image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(valid_extensions)]

    if not image_files:
        print("資料夾中沒有找到支援的圖片。")
        return

    print(f"找到 {len(image_files)} 張圖片，開始進行 Windows Native OCR 提取...\n")

    with open(output_file, "w", encoding="utf-8") as out:
        for filename in image_files:
            image_path = os.path.join(folder_path, filename)
            try:
                extracted_text = await run_win_ocr_on_file(ocr_engine, image_path)
                
                print(f"=== 檔案：{filename} ===")
                print(extracted_text.strip() if extracted_text.strip() else "(未辨識出文字)")
                print("-" * 40)
                
                out.write(f"=== 檔案：{filename} ===\n")
                out.write(extracted_text.strip() + "\n\n")

            except Exception as e:
                print(f"處理 {filename} 時發生錯誤: {e}")

    print(f"\n所有文字已成功提取並儲存至：{output_file}")

# 在 Jupyter Notebook (.ipynb) 中直接 await 呼叫
await main()